# Sprint 3 - Baseline Models (NER + ABSA)

**Input:** output Sprint 2 di `data/processed/`
- NER: `ner_{train,val,test}.jsonl`
- ABSA: `absa_{train,val,test}.csv`

**Tujuan:** menetapkan **referensi performa** non-transformer. Angka di sini adalah patokan yang harus dikalahkan oleh model transformer di Sprint 4.

**Kernel: Python 3.14** (yang punya sklearn, sklearn_crfsuite, seqeval).

---
- **Part A - NER:** Conditional Random Field (CRF). Memodelkan P(seluruh tag sequence | seluruh token sequence) -> menangkap dependensi antar-label (`I-ENT` butuh `B-ENT` sebelumnya).
- **Part B - ABSA:** TF-IDF + Linear SVM. Catatan: BoW kehilangan posisi -> tidak benar-benar aspect-aware. Kita ukur keterbatasan ini secara kuantitatif.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from collections import Counter

PROJECT_ROOT   = Path('..').resolve()
DATA_PROCESSED = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR      = PROJECT_ROOT / 'backend' / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print('Data :', DATA_PROCESSED)
print('Model:', MODEL_DIR)

Data : D:\Text Mining Projects\Financial-News-Miner\data\processed
Model: D:\Text Mining Projects\Financial-News-Miner\backend\models


## PART A - NER Baseline (CRF)

### A.1 Load data NER

In [2]:
def load_ner(path):
    sents, labels = [], []
    with open(path, encoding='utf-8') as f:
        for line in f:
            d = json.loads(line)
            sents.append(d['tokens'])
            labels.append(d['ner_tags'])
    return sents, labels

train_sents, train_labels = load_ner(DATA_PROCESSED / 'ner_train.jsonl')
val_sents,   val_labels   = load_ner(DATA_PROCESSED / 'ner_val.jsonl')
test_sents,  test_labels  = load_ner(DATA_PROCESSED / 'ner_test.jsonl')

print(f'train: {len(train_sents)} headlines')
print(f'val  : {len(val_sents)} headlines')
print(f'test : {len(test_sents)} headlines')
print('\nLabel distribution (train tokens):',
      Counter(t for seq in train_labels for t in seq))

train: 8509 headlines
val  : 1063 headlines
test : 1063 headlines

Label distribution (train tokens): Counter({'O': 66533, 'B-ENT': 11376, 'I-ENT': 6416})


### A.2 Feature engineering

Tiap token jadi dict fitur. CRF tidak butuh fitur numerik padat seperti SVM — dia menerima fitur kategorikal/boolean sebagai sparse indicator.

**Fitur per token:** bentuk kata (`isupper`, `istitle`, `isdigit`), prefix/suffix, ada-tidaknya digit/hyphen, panjang. **Konteks window [-2, +2]:** fitur tetangga, karena entity sering ditentukan oleh konteks ("YES Bank" — kata "Bank" menandakan token sebelumnya bagian entitas).

Mengapa `istitle`/`isupper` penting di sini: entitas finansial hampir selalu proper noun yang dikapitalisasi (`BSE`, `Nifty50`) — fitur ini sangat diskriminatif untuk dataset headline.

In [3]:
def word2features(sent, i):
    w = sent[i]
    f = {
        'bias': 1.0,
        'w.lower': w.lower(),
        'w.suffix3': w[-3:],
        'w.suffix2': w[-2:],
        'w.prefix2': w[:2],
        'w.isupper': w.isupper(),
        'w.istitle': w.istitle(),
        'w.isdigit': w.isdigit(),
        'w.hasdigit': any(c.isdigit() for c in w),
        'w.hashyphen': '-' in w,
        'w.len': len(w),
    }
    # konteks kiri
    if i > 0:
        p = sent[i-1]
        f.update({'-1.lower': p.lower(), '-1.istitle': p.istitle(), '-1.isupper': p.isupper()})
    else:
        f['BOS'] = True
    if i > 1:
        p = sent[i-2]
        f.update({'-2.lower': p.lower(), '-2.istitle': p.istitle()})
    # konteks kanan
    if i < len(sent) - 1:
        n = sent[i+1]
        f.update({'+1.lower': n.lower(), '+1.istitle': n.istitle(), '+1.isupper': n.isupper()})
    else:
        f['EOS'] = True
    if i < len(sent) - 2:
        n = sent[i+2]
        f.update({'+2.lower': n.lower(), '+2.istitle': n.istitle()})
    return f

def sent2features(sent):
    return [word2features(sent, i) for i in range(len(sent))]

X_train = [sent2features(s) for s in train_sents]
X_val   = [sent2features(s) for s in val_sents]
X_test  = [sent2features(s) for s in test_sents]
print('Contoh fitur token pertama headline pertama:')
for k, v in list(X_train[0][0].items())[:8]:
    print(f'  {k}: {v}')

Contoh fitur token pertama headline pertama:
  bias: 1.0
  w.lower: mmtc
  w.suffix3: MTC
  w.suffix2: TC
  w.prefix2: MM
  w.isupper: True
  w.istitle: False
  w.isdigit: False


### A.3 Train CRF

`c1`/`c2` = regularisasi L1/L2 (analog Lasso/Ridge). `all_possible_transitions=True` membiarkan CRF belajar transisi yang tidak muncul di train (mis. melarang `O -> I-ENT`). Nilai c1/c2 default-ish; bisa di-tune di val set kalau ada waktu.

In [4]:
from sklearn_crfsuite import CRF

crf = CRF(
    algorithm='lbfgs',
    c1=0.1, c2=0.1,
    max_iterations=100,
    all_possible_transitions=True,
)
crf.fit(X_train, train_labels)
print('CRF trained. #features:', len(crf.state_features_), '| #transitions:', len(crf.transition_features_))

CRF trained. #features: 26795 | #transitions: 9


### A.4 Evaluasi (seqeval - entity-level F1)

**Penting:** `seqeval` menghitung F1 di level **span entitas**, bukan per-token. Artinya prediksi "YES" benar tapi "Bank" salah = entity "YES Bank" dihitung **salah total**. Ini metric yang jujur untuk NER (berbeda dari token-accuracy yang menggelembung karena mayoritas token = `O`).

In [5]:
from seqeval.metrics import classification_report as seq_report, f1_score as seq_f1

y_pred_test = crf.predict(X_test)
print('=== NER CRF - TEST set (entity-level) ===')
print(seq_report(test_labels, y_pred_test))
print('Entity-level F1 (micro):', round(seq_f1(test_labels, y_pred_test), 4))

=== NER CRF - TEST set (entity-level) ===
              precision    recall  f1-score   support

         ENT       0.85      0.82      0.84      1437

   micro avg       0.85      0.82      0.84      1437
   macro avg       0.85      0.82      0.84      1437
weighted avg       0.85      0.82      0.84      1437

Entity-level F1 (micro): 0.8365


### A.5 Interpretabilitas - transisi & fitur teratas

Salah satu kelebihan CRF: bisa dibaca. Mari lihat transisi label yang dipelajari dan fitur paling berpengaruh.

In [6]:
print('Top transitions (from -> to : weight):')
trans = Counter(crf.transition_features_).most_common(6)
for (a, b), w in trans:
    print(f'  {a:6s} -> {b:6s} : {w:+.2f}')

print('\nTop state features (label : feature : weight):')
states = Counter(crf.state_features_).most_common(12)
for (feat, label), w in states:
    print(f'  {label:6s} : {feat:24s} : {w:+.2f}')

Top transitions (from -> to : weight):
  B-ENT  -> I-ENT  : +1.94
  O      -> O      : +1.80
  I-ENT  -> I-ENT  : +0.07
  O      -> B-ENT  : -0.01
  B-ENT  -> O      : -0.96
  I-ENT  -> O      : -1.19

Top state features (label : feature : weight):
  B-ENT  : w.lower:sterling         : +8.26
  B-ENT  : w.lower:brent            : +5.46
  B-ENT  : w.lower:pound            : +4.41
  B-ENT  : -1.lower:euro,           : +4.35
  B-ENT  : -1.lower:prefer          : +4.33
  B-ENT  : BOS                      : +4.14
  B-ENT  : w.prefix2:Ci             : +4.14
  B-ENT  : w.lower:reliance         : +4.04
  B-ENT  : w.lower:silver           : +4.03
  I-ENT  : -1.lower:stock           : +3.99
  O      : w.lower:sell             : +3.96
  B-ENT  : -1.lower:sensex,         : +3.88


### A.6 Error analysis - contoh prediksi salah

In [7]:
def show_errors(sents, gold, pred, n=6):
    shown = 0
    for s, g, p in zip(sents, gold, pred):
        if g != p:
            print('TOKENS:', ' '.join(s))
            diff = [(tok, gt, pt) for tok, gt, pt in zip(s, g, p) if gt != pt]
            for tok, gt, pt in diff:
                print(f'   {tok:18s} gold={gt:6s} pred={pt:6s}')
            print()
            shown += 1
            if shown >= n:
                break

show_errors(test_sents, test_labels, y_pred_test)

TOKENS: Billionaire's stake shakes Woolworth's buyout of David Jones
   Billionaire's      gold=O      pred=B-ENT 
   David              gold=B-ENT  pred=O     
   Jones              gold=I-ENT  pred=O     

TOKENS: How will Shell-BG deal impact global energy market
   Shell-BG           gold=B-ENT  pred=O     

TOKENS: Ravi Venkatesan joins Rockefeller Foundation Board
   Board              gold=I-ENT  pred=O     

TOKENS: Stay with quality stocks: Deven Choksey
   quality            gold=B-ENT  pred=O     
   stocks:            gold=I-ENT  pred=O     

TOKENS: January 11, 2014: Mecklai Financial report on rupee
   rupee              gold=O      pred=B-ENT 

TOKENS: General Insurance Corporation to offer catastrophe bonds
   catastrophe        gold=O      pred=B-ENT 
   bonds              gold=O      pred=I-ENT 



In [8]:
joblib.dump(crf, MODEL_DIR / 'ner_crf_baseline.joblib')
print('Saved:', MODEL_DIR / 'ner_crf_baseline.joblib')

Saved: D:\Text Mining Projects\Financial-News-Miner\backend\models\ner_crf_baseline.joblib


## PART B - ABSA Baseline (TF-IDF + Linear SVM)

### B.1 Load data & bentuk input

Input = `title + " [SEP] " + entity`. **Keterbatasan yang diukur nanti:** BoW kehilangan posisi, jadi untuk headline multi-entity dengan sentimen bertentangan, fitur title-nya identik untuk tiap entitas. Baseline ini akan kesulitan di sana — dan itu yang kita kuantifikasi.

In [9]:
absa_train = pd.read_csv(DATA_PROCESSED / 'absa_train.csv')
absa_test  = pd.read_csv(DATA_PROCESSED / 'absa_test.csv')

for d in (absa_train, absa_test):
    d['text'] = d['title'] + ' [SEP] ' + d['entity']

print('train pairs:', len(absa_train), '| test pairs:', len(absa_test))
print('\nLabel map: 0=negative, 1=neutral, 2=positive')
print('Train label dist:', Counter(absa_train['label']))

train pairs: 11433 | test pairs: 1448

Label map: 0=negative, 1=neutral, 2=positive
Train label dist: Counter({1: 4341, 2: 4048, 0: 3044})


### B.2 Train TF-IDF + LinearSVC

`ngram_range=(1,2)` menangkap bigram seperti "net loss", "record high". `class_weight='balanced'` menangani imbalance (negative paling sedikit) dengan membobot loss berbanding terbalik dengan frekuensi kelas — bukan SMOTE.

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC

vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_tr = vectorizer.fit_transform(absa_train['text'])
X_te = vectorizer.transform(absa_test['text'])
print('TF-IDF matrix:', X_tr.shape, '(n_pairs x n_features)')

svm = LinearSVC(class_weight='balanced', C=1.0, max_iter=5000)
svm.fit(X_tr, absa_train['label'])
print('SVM trained.')

TF-IDF matrix: (11433, 27051) (n_pairs x n_features)
SVM trained.


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

pred_te = svm.predict(X_te)
names = ['negative', 'neutral', 'positive']
print('=== ABSA TF-IDF+SVM - TEST set ===')
print(classification_report(absa_test['label'], pred_te, target_names=names, digits=4))

cm = confusion_matrix(absa_test['label'], pred_te)
print('Confusion matrix (rows=gold, cols=pred):')
print(pd.DataFrame(cm, index=[f'true_{n}' for n in names], columns=[f'pred_{n}' for n in names]))

=== ABSA TF-IDF+SVM - TEST set ===
              precision    recall  f1-score   support

    negative     0.7686    0.7266    0.7470       384
     neutral     0.7744    0.7956    0.7849       548
    positive     0.7644    0.7733    0.7688       516

    accuracy                         0.7693      1448
   macro avg     0.7691    0.7651    0.7669      1448
weighted avg     0.7693    0.7693    0.7691      1448

Confusion matrix (rows=gold, cols=pred):
               pred_negative  pred_neutral  pred_positive
true_negative            279            46             59
true_neutral              48           436             64
true_positive             36            81            399


### B.3 Error analysis kunci - single-entity vs multi-entity

Inilah pengukuran yang memotivasi transformer. Kita pecah akurasi test berdasarkan:
- **single:** headline dengan 1 entitas (baseline harusnya OK)
- **multi-uniform:** headline >1 entitas tapi semua sentimen sama
- **multi-conflict:** headline >1 entitas dengan sentimen berbeda (baseline harusnya GAGAL — BoW tidak bisa bedakan)

Jika akurasi `multi-conflict` jauh di bawah `single`, itu bukti empiris keterbatasan BoW untuk ABSA.

In [12]:
# Kategorikan tiap pasangan test berdasarkan tipe headline-nya
absa_test = absa_test.copy()
absa_test['pred'] = pred_te
absa_test['correct'] = (absa_test['pred'] == absa_test['label'])

grp = absa_test.groupby('s_no')
n_ent = grp['entity'].transform('size')
n_sent = grp['sentiment'].transform('nunique')

def categorize(ne, ns):
    if ne == 1: return 'single'
    return 'multi-conflict' if ns > 1 else 'multi-uniform'

absa_test['head_type'] = [categorize(ne, ns) for ne, ns in zip(n_ent, n_sent)]

summary = absa_test.groupby('head_type').agg(
    n_pairs=('correct', 'size'),
    accuracy=('correct', 'mean'),
).round(4)
print('Akurasi per tipe headline:')
print(summary)
print('\nGap single vs multi-conflict =',
      round(summary.loc['single','accuracy'] - summary.loc['multi-conflict','accuracy'], 4))

Akurasi per tipe headline:
                n_pairs  accuracy
head_type                        
multi-conflict      294    0.6156
multi-uniform       376    0.7819
single              778    0.8213

Gap single vs multi-conflict = 0.2057


In [13]:
# Contoh kesalahan di multi-conflict (jantung kelemahan BoW)
print('Contoh kesalahan di headline multi-conflict:')
err = absa_test[(absa_test['head_type'] == 'multi-conflict') & (~absa_test['correct'])]
id2name = {0: 'neg', 1: 'neu', 2: 'pos'}
for sno in err['s_no'].unique()[:5]:
    g = absa_test[absa_test['s_no'] == sno]
    print('  TITLE:', g['title'].iloc[0])
    for _, r in g.iterrows():
        mark = 'OK ' if r['correct'] else 'XX '
        print(f'     {mark} {r["entity"]:22s} gold={id2name[r["label"]]} pred={id2name[r["pred"]]}')
    print()

Contoh kesalahan di headline multi-conflict:
  TITLE: Sterling climbs against Euro on diverging policy outlooks
     OK  Sterling               gold=pos pred=pos
     XX  Euro                   gold=neu pred=pos

  TITLE: Nikkei rises as yen plumbs 6-year lows
     OK  Nikkei                 gold=pos pred=pos
     XX  yen                    gold=neg pred=pos

  TITLE: Export-oriented cos to outdo domestic cyclicals: Hemindra Hazari
     XX  Export-oriented cos    gold=pos pred=neg
     OK  domestic cyclicals     gold=neg pred=neg

  TITLE: Asian shares prove resilient to euro jitters
     XX  Asian shares           gold=pos pred=neg
     OK  euro                   gold=neg pred=neg

  TITLE: Orchid Chemicals gain 5% on milestone payment from Merck
     OK  Orchid Chemicals       gold=pos pred=pos
     XX  Merck                  gold=neu pred=pos



In [14]:
joblib.dump({'vectorizer': vectorizer, 'svm': svm},
            MODEL_DIR / 'absa_tfidf_svm_baseline.joblib')
print('Saved:', MODEL_DIR / 'absa_tfidf_svm_baseline.joblib')

Saved: D:\Text Mining Projects\Financial-News-Miner\backend\models\absa_tfidf_svm_baseline.joblib


## Ringkasan Sprint 3 (isi setelah run)

**NER CRF:**
- [ ] Entity-level F1 (test): ___

**ABSA TF-IDF+SVM:**
- [ ] Macro-F1 (test): ___
- [ ] F1 per kelas: neg ___ / neu ___ / pos ___
- [ ] Akurasi single ___ vs multi-conflict ___ -> gap ___

**Insight untuk Sprint 4:** gap multi-conflict mengkonfirmasi keterbatasan BoW -> transformer dengan input `[CLS] title [SEP] entity [SEP]` diharapkan menutup gap ini.